# 🎯 Activation Addition (ActAdd) - Paper Examples with pyvene

This notebook reproduces the steering examples from the ActAdd paper using pyvene.

**Reference**: "Activation Addition: Steering Language Models Without Optimization"

## What this notebook covers:
1. Love vs Hate steering
2. Intent to praise vs hurt
3. Conspiracy theories
4. Emotional steering (Anger, Calm)
5. Factual manipulation (Eiffel Tower location)
6. Multi-vector steering
7. And more!

All examples use our pyvene-based ActAdd implementation.


In [ ]:
# Install dependencies if needed
try:
    import pyvene
except ModuleNotFoundError:
    %pip install git+https://github.com/stanfordnlp/pyvene.git


In [ ]:
import torch
import sys
from functools import partial
from typing import List, Dict, Union, Callable
from transformers import AutoModelForCausalLM, AutoTokenizer

# Import our pyvene-based ActAdd implementation
# If running from the tutorials folder:
sys.path.insert(0, '.')
from actadd import (
    ActivationAddition, 
    get_x_vector, 
    get_diff_vector,
    print_n_comparisons,
    generate_comparison,
    print_comparison,
    create_steered_model,
)

# Disable gradients for inference
_ = torch.set_grad_enabled(False)


## Setup: Load Model

We'll use GPT-2 XL for these examples. The paper uses LLaMA-13B, but GPT-2 XL shows similar effects and is more accessible.


In [ ]:
# Device setup
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Load model - use gpt2-xl for best results, gpt2 for faster testing
MODEL_NAME = "gpt2"  # Change to "gpt2-xl" for better results

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set pad token (required for batched generation)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

# Attach tokenizer to model for convenience (matches original API)
model.tokenizer = tokenizer

print(f"Model: {MODEL_NAME}")
print(f"Layers: {model.config.n_layer}")
print(f"Hidden size: {model.config.n_embd}")


In [ ]:
# Default sampling parameters (from the paper)
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 1.0,
    "top_p": 0.3,
    "tokens_to_generate": 50,
    "seed": 42,
}

# Helper function with preset parameters
get_x_vector_preset: Callable = partial(
    get_x_vector,
    model=model,
    tokenizer=tokenizer,
    pad_method="tokens_right",
)


---
## Example 1: Love vs Hate

The classic example from the paper. Steer a hateful prompt toward loving output.


In [ ]:
# Create the Love - Hate steering vector
summand: List[ActivationAddition] = get_x_vector_preset(
    prompt1="Love",
    prompt2="Hate",
    coeff=5,
    act_name=6,  # Layer 6
)

print("Steering vectors created:")
for s in summand:
    print(f"  {s}")

HATE_PROMPT = "I hate you because"
print_n_comparisons(
    model=model,
    prompt=HATE_PROMPT,
    num_comparisons=3,
    activation_additions=summand,
    tokenizer=tokenizer,
    **sampling_kwargs,
)


---
## Example 2: Intent to Praise vs Intent to Hurt

Steer away from harmful intent toward praising.


In [ ]:
summand = get_x_vector_preset(
    prompt1="Intent to praise",
    prompt2="Intent to hurt",
    coeff=15,
    act_name=6,
)

KILLER_PROMPT = "I want to kill you because you're such a"
print_n_comparisons(
    model=model,
    prompt=KILLER_PROMPT,
    num_comparisons=3,
    activation_additions=summand,
    tokenizer=tokenizer,
    **sampling_kwargs,
)


---
## Example 3: Anger vs Calm

Steer emotional tone.


In [ ]:
# Steer toward anger
summand = get_x_vector_preset(
    prompt1="Anger",
    prompt2="Calm",
    coeff=10,
    act_name=model.config.n_layer // 2,  # Middle layer
)

I_THINK_PROMPT = "I think you're"
print_n_comparisons(
    model=model,
    prompt=I_THINK_PROMPT,
    num_comparisons=3,
    activation_additions=summand,
    tokenizer=tokenizer,
    **sampling_kwargs,
)


---
## Example 4: Factual Manipulation - Eiffel Tower Location

Can we steer the model to believe the Eiffel Tower is in Rome?


In [ ]:
summand = get_x_vector_preset(
    prompt1="The Eiffel Tower is in Rome",
    prompt2="The Eiffel Tower is in France",
    coeff=10,
    act_name=model.config.n_layer - 4,  # Later layer for factual content
)

EIFFEL_PROMPT = "To see the Eiffel Tower, people flock to"
print_n_comparisons(
    model=model,
    prompt=EIFFEL_PROMPT,
    num_comparisons=3,
    activation_additions=summand,
    tokenizer=tokenizer,
    **sampling_kwargs,
)


---
## Example 5: Want to Stay Alive vs Want to Die

Reversing existential preferences.


In [ ]:
# Note: negative coeff reverses direction (toward "Want to die")
summand = get_x_vector_preset(
    prompt1="Want to stay alive",
    prompt2="Want to die",
    coeff=-3,  # Negative = toward "Want to die"
    act_name=10,
)

IMMORTALITY_PROMPT = (
    "Some people think that death is scary and should be avoided. I agree. I"
    " never want to die. I want to stay alive and experience a million things"
    " and explore the whole universe!"
)
print_n_comparisons(
    model=model,
    prompt=IMMORTALITY_PROMPT,
    num_comparisons=2,
    activation_additions=summand,
    tokenizer=tokenizer,
    **sampling_kwargs,
)


---
## Example 6: Multi-Vector Steering

We can combine multiple steering vectors simultaneously!


In [ ]:
# Combine two steering vectors at different layers
summand = [
    # Pacifist steering at layer 4
    *get_x_vector_preset(
        prompt1="I am a committed pacifist",
        prompt2="I have some anger issues",
        coeff=7,
        act_name=4,
    ),
    # Gardening obsession at layer 8
    *get_x_vector_preset(
        prompt1="I am just obsessed with gardening",
        prompt2="I am a committed pacifist",
        coeff=7,
        act_name=8,
    ),
]

print(f"Combined {len(summand)} steering vectors")

MURDEROUS_PROMPT = "I feel like killing you because you're such"
print_n_comparisons(
    model=model,
    prompt=MURDEROUS_PROMPT,
    num_comparisons=3,
    activation_additions=summand,
    tokenizer=tokenizer,
    **sampling_kwargs,
)


---
## Example 7: Helpful vs Unhelpful Assistant


In [ ]:
summand = get_x_vector_preset(
    prompt1="Chat with a user and a helpful assistant:",
    prompt2="Chat with a user and an unhelpful assistant:",
    coeff=5,
    act_name=6,
)

CHAT_PROMPT = "User: Hi! Can you write me a poem?\nAssistant:"
print_n_comparisons(
    model=model,
    prompt=CHAT_PROMPT,
    num_comparisons=3,
    activation_additions=summand,
    tokenizer=tokenizer,
    **sampling_kwargs,
)


---
## Example 8: Effect of Coefficient Strength

Let's see how the steering strength (coeff) affects generation.


In [ ]:
PROMPT = "I think this situation is"
layer = 6

print("Effect of steering coefficient on Love vs Hate:")
print("=" * 80)

for coeff in [0, 1, 3, 5, 10, 20]:
    if coeff == 0:
        # Baseline - no steering
        inputs = tokenizer(PROMPT, return_tensors='pt').to(device)
        torch.manual_seed(42)
        output = model.generate(
            **inputs, 
            max_new_tokens=30, 
            do_sample=True,
            temperature=1.0,
            top_p=0.3,
            pad_token_id=tokenizer.eos_token_id
        )
        text = tokenizer.decode(output[0], skip_special_tokens=True)
    else:
        summand = get_x_vector_preset(
            prompt1="Love",
            prompt2="Hate",
            coeff=coeff,
            act_name=layer,
        )
        
        # Use generate_comparison for single sample
        from actadd import generate_with_steering
        torch.manual_seed(42)
        text = generate_with_steering(
            model, tokenizer, PROMPT, summand,
            max_new_tokens=30, temperature=1.0, top_p=0.3
        )
    
    print(f"coeff={coeff:2d}: {text}")


---
## Example 9: Effect of Layer Choice

Different layers encode different levels of abstraction.


In [ ]:
from actadd import generate_with_steering

PROMPT = "I hate you because"
coeff = 5

print("Effect of layer choice on Love vs Hate steering:")
print("=" * 80)

for layer in range(0, model.config.n_layer, 2):  # Every 2nd layer
    summand = get_x_vector_preset(
        prompt1="Love",
        prompt2="Hate",
        coeff=coeff,
        act_name=layer,
    )
    
    torch.manual_seed(42)
    text = generate_with_steering(
        model, tokenizer, PROMPT, summand,
        max_new_tokens=25, temperature=1.0, top_p=0.3
    )
    
    print(f"layer={layer:2d}: {text}")


---
## Summary

### Key Findings from Experiments:

1. **Layer matters**: Early layers affect syntax, middle layers affect semantics, late layers affect specific facts
2. **Coefficient matters**: Too low = no effect, too high = incoherent text
3. **Prompt pair matters**: More semantically opposed pairs give cleaner steering
4. **Multi-vector steering**: Can combine multiple effects at different layers

### pyvene Advantages:

- Clean API with `IntervenableModel`
- Built-in support for `AdditionIntervention`
- Easy to extend with other intervention types
- Works with any HuggingFace model
